# A driven damped oscillator: the route to chaos

The Duffing oscillator $\ddot x + \delta \dot x - x + x^3 = F\cos\omega t$ is a particle in a double-well potential,
damped and periodically shaken. It is built here entirely from built-in forces (two `PowerLaw` terms for the
potential, `LinearDrag`, `PeriodicForce`), so it runs in Rust. With $\delta = 0.3$, $\omega = 1.2$, raising the drive
$F$ takes the motion through period doubling into chaos. Sampling $x$ once per drive period (a stroboscopic Poincaré
map) shows it: one point for period-1 motion, two after the first doubling, a smear for chaos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import physim as ps

In [ ]:
delta, omega = 0.3, 1.2
period, per = 2 * np.pi / omega, 100        # 100 steps per drive period

def duffing(F, x0=0.5):
    w = ps.World("rk4")                     # velocity-dependent forces: use rk4
    w.add_particle([x0, 0, 0])
    w.add_force(ps.PowerLaw(-0.5, 2.0))     # -x^2/2
    w.add_force(ps.PowerLaw(0.25, 4.0))     # +x^4/4
    w.add_force(ps.LinearDrag(delta))
    w.add_force(ps.PeriodicForce(0, [F, 0, 0], omega))
    return w

def strobe(F, transient=200, keep=60):
    w = duffing(F)
    w.run(period / per, transient * per, record_every=transient * per, energies=False)
    return w.run(period / per, keep * per, record_every=per, energies=False).pos[1:, 0, 0]

forces = np.linspace(0.2, 0.42, 300)
fig, ax = plt.subplots(figsize=(8, 4))
for F in forces:
    x = strobe(F)
    ax.plot(np.full_like(x, F), x, ",", color=ps.plot.color(0), alpha=0.6)
ax.set_xlabel("drive amplitude F"); ax.set_ylabel("x at each drive period"); ps.plot.tidy(ax);

At $F = 0.5$ the motion is chaotic and the stroboscopic points fill a fractal **strange attractor**. Its largest
Lyapunov exponent is positive.

In [ ]:
w = duffing(0.5)
w.run(period / per, 100 * per, record_every=100 * per, energies=False)
tr = w.run(period / per, 5000 * per, record_every=per, energies=False)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(tr.pos[:, 0, 0], tr.vel[:, 0, 0], ".", ms=0.8, color=ps.plot.color(0))
ax.set_xlabel("x"); ax.set_ylabel("v"); ax.set_title("Duffing attractor, F = 0.5"); ps.plot.tidy(ax)
lam = duffing(0.5).lyapunov(period / per, 400 * per, n=1, record_every=400 * per)["exponents"][0]
print(f"largest Lyapunov exponent at F = 0.5: {lam:.3f} per unit time")